In [ ]:
!pip install python-binance
!pip install python-dotenv
!pip install jupyter_contrib_nbextensions

In [ ]:
from dotenv import load_dotenv
import os
from binance.client import Client
load_dotenv(override=True)# Carga el archivo .env
api_key = os.getenv('BINANCE_API_KEY')
api_secret = os.getenv('BINANCE_API_SECRET')
client = Client(api_key, api_secret)
#imprimo todo mi balance, suipongo que es de la billetera spot
print(client.get_account()["balances"],type(client.get_account()["balances"]))

In [ ]:
balances = client.get_account()["balances"]
btc_balance = next(b for b in balances if b['asset'] == 'BTC')
usdt_balance = next(b for b in balances if b['asset'] == 'USDT')
print("BTC disponible:", btc_balance['free'])
print("USDT disponible:", usdt_balance['free'])

In [ ]:
ticker = client.get_symbol_ticker(symbol="BTCUSDT")
print("Precio actual BTC/USDT:", ticker["price"])


In [ ]:
from decimal import Decimal, ROUND_DOWN
step_size = '0.00001000'
min_qty = 0.00001

# Función para redondear al múltiplo válido de stepSize
def ajustar_step_size(cantidad, step_size_str):
    step = Decimal(step_size_str)
    cantidad_decimal = Decimal(str(cantidad))
    return float(cantidad_decimal.quantize(step, rounding=ROUND_DOWN))


In [ ]:
#vendo la mitad de lo que tengo en btc a ustd
btc_disponible = float(btc_balance['free'])
print(btc_disponible)
btc_a_vender = round(btc_disponible*0.5,6)
btc_a_vender = ajustar_step_size(btc_a_vender, step_size)

print(btc_a_vender)


In [ ]:

# Verificar que sea mayor al mínimo
if btc_a_vender >= min_qty:
    orden = client.order_market_sell(symbol='BTCUSDT', quantity=btc_a_vender)
    print(f"Orden ejecutada: {orden}")
else:
    print(f"No se puede vender: cantidad ajustada ({btc_a_vender}) menor al mínimo permitido ({min_qty}).")


In [ ]:
from decimal import Decimal, ROUND_DOWN

# Función para redondear al múltiplo válido de stepSize
def ajustar_step_size(cantidad, step_size_str):
    step = Decimal(step_size_str)
    cantidad_decimal = Decimal(str(cantidad))
    return float(cantidad_decimal.quantize(step, rounding=ROUND_DOWN))

# Parámetros del filtro
step_size = '0.00001000'
min_qty = 0.00001

# Obtener BTC disponible
btc_balance = next(b for b in client.get_account()['balances'] if b['asset'] == 'BTC')
btc_disponible = float(btc_balance['free'])
print(btc_disponible)
# Calcular la mitad
btc_a_vender = btc_disponible / 2

# Ajustar al múltiplo válido
btc_a_vender = ajustar_step_size(btc_a_vender, step_size)

# Verificar que sea mayor al mínimo
if btc_a_vender >= min_qty:
    orden = client.order_market_sell(symbol='BTCUSDT', quantity=btc_a_vender)
    print(f"Orden ejecutada: {orden}")
else:
    print(f"No se puede vender: cantidad ajustada ({btc_a_vender}) menor al mínimo permitido ({min_qty}).")


In [ ]:
# Por ejemplo, para el par BTCUSDT
trades = client.get_my_trades(symbol='BTCUSDT')

for trade in trades:
    print(f"ID: {trade['id']}, Precio: {trade['price']}, Cantidad: {trade['qty']}, Tiempo: {trade['time']}, Comisión: {trade['commission']} {trade['commissionAsset']}")


In [ ]:
balances = client.get_account()['balances']
for b in balances:
    if float(b['free']) > 0:
        print(f"{b['asset']}: {b['free']} disponibles")


In [ ]:
symbol: str = "BTCUSDT"
ticker = client.get_symbol_ticker(symbol=symbol)
print(ticker)
ticker["price"]


In [ ]:
from binance.client import Client
from binance.exceptions import BinanceAPIException
import os
from dotenv import load_dotenv
import numpy as np
from scipy.stats import lognorm
import time
from decimal import Decimal, ROUND_DOWN

# ======= CONFIGURACIÓN =======
load_dotenv()
api_key = os.getenv("BINANCE_API_KEY")
api_secret = os.getenv("BINANCE_API_SECRET")
client = Client(api_key, api_secret)

INTERVALO_ESPERA = 60  # segundos entre chequeos
MONTO_USDT_POR_COMPRA = 50  # USDT a usar en cada compra
CANTIDAD_MINIMA_POR_VENTA = 0.001  # BTC o token mínimo para vender (ajustar por token)

# ======= FUNCIONES UTILITARIAS =======

def ajustar_step_size(cantidad, step_size_str):
    step = Decimal(step_size_str)
    cantidad_decimal = Decimal(str(cantidad))
    cantidad_floor = (cantidad_decimal // step) * step
    return float(cantidad_floor)

def obtener_step_min_qty(symbol):
    info = client.get_symbol_info(symbol)
    filtro = next(f for f in info['filters'] if f['filterType'] == 'LOT_SIZE')
    return filtro['stepSize'], float(filtro['minQty'])

def obtener_activos_con_saldo():
    balances = client.get_account()['balances']
    return [b['asset'] for b in balances if float(b['free']) > 0]

def obtener_pares_disponibles():
    info = client.get_exchange_info()
    return [s['symbol'] for s in info['symbols']]

def pares_usdt_activos(activos, pares):
    pares_validos = []
    for asset in activos:
        simbolo = asset + "USDT"
        if simbolo in pares:
            pares_validos.append(simbolo)
    return pares_validos

def obtener_precios_historicos(symbol, dias=90):
    klines = client.get_historical_klines(symbol, Client.KLINE_INTERVAL_1DAY, f"{dias} days ago UTC")
    return [float(k[4]) for k in klines]  # precio de cierre

def obtener_saldo(asset):
    balances = client.get_account()['balances']
    bal = next((b for b in balances if b['asset'] == asset), None)
    return float(bal['free']) if bal else 0.0

# ======= FUNCIONES DE COMPRA Y VENTA GENERALIZADAS =======

def comprar_asset_usdt(symbol, monto_usdt):
    base_asset = symbol.replace("USDT", "")
    step_size_str, min_qty = obtener_step_min_qty(symbol)
    saldo_usdt = obtener_saldo("USDT")

    monto_real = min(monto_usdt, saldo_usdt)
    if monto_real < 10:
        print(f"⚠️ Saldo USDT insuficiente para comprar en {symbol}: {monto_real:.2f} USDT")
        return

    precio = float(client.get_symbol_ticker(symbol=symbol)["price"])
    cantidad = ajustar_step_size(monto_real / precio, step_size_str)

    if cantidad >= min_qty:
        try:
            orden = client.order_market_buy(symbol=symbol, quantity=cantidad)
            print(f"🟢✅ COMPRA ejecutada en {symbol}")
            print(f"🔺 Cantidad: {cantidad} {base_asset} | Precio estimado: {precio:.2f} USDT")
            print(f"🔁 Detalles de orden: {orden}")
        except BinanceAPIException as e:
            print(f"❌ Error al comprar en {symbol}:", e)
    else:
        print(f"⚠️ Cantidad calculada ({cantidad}) menor al mínimo permitido ({min_qty}) para {symbol}")

def vender_asset(symbol, cantidad_deseada):
    base_asset = symbol.replace("USDT", "")
    step_size_str, min_qty = obtener_step_min_qty(symbol)
    saldo_disponible = obtener_saldo(base_asset)

    cantidad_real = min(cantidad_deseada, saldo_disponible)
    cantidad_ajustada = ajustar_step_size(cantidad_real, step_size_str)

    if cantidad_ajustada >= min_qty:
        try:
            orden = client.order_market_sell(symbol=symbol, quantity=cantidad_ajustada)
            print(f"🔴✅ VENTA ejecutada en {symbol}")
            print(f"🔻 Vendido: {cantidad_ajustada} {base_asset} | Saldo disponible: {saldo_disponible}")
            print(f"🔁 Detalles de orden: {orden}")
        except BinanceAPIException as e:
            print(f"❌ Error al vender en {symbol}:", e)
    else:
        print(f"⚠️ No se puede vender {cantidad_ajustada} < mínimo requerido ({min_qty}) en {symbol}")

# ======= FUNCIÓN DECISORIA GENERALIZADA =======

def decidir_y_actuar(precio_actual, precios, shape, loc, scale, symbol):
    media = np.mean(precios)
    std = np.std(precios)
    z = (precio_actual - media) / std

    boll_std_mult = 1.5
    boll_upper = media + boll_std_mult * std
    boll_lower = media - boll_std_mult * std

    p20 = np.exp(lognorm.ppf(0.20, shape, loc=loc, scale=scale))
    p80 = np.exp(lognorm.ppf(0.80, shape, loc=loc, scale=scale))

    print(f"\n[{symbol}] Precio actual: {precio_actual:.6f}")
    print(f"Log-normal percentiles: P20={p20:.6f}, P80={p80:.6f}")
    print(f"Z-score: {z:.2f}")
    print(f"Bollinger Bands: [ {boll_lower:.6f} , {boll_upper:.6f} ]")

    base_asset = symbol.replace("USDT", "")
    cantidad_minima_venta = 0.001  # Ajusta según token si querés precisión

    if (
        precio_actual < p20 or
        z <= -1.0 or
        precio_actual < boll_lower
    ):
        print(f"🟢 Señal de COMPRA detectada en {symbol}.")
        comprar_asset_usdt(symbol, MONTO_USDT_POR_COMPRA)

    elif (
        precio_actual > p80 or
        z >= 1.0 or
        precio_actual > boll_upper
    ):
        print(f"🔴 Señal de VENTA detectada en {symbol}.")
        vender_asset(symbol, cantidad_minima_venta)

    else:
        print(f"⏸️ Sin señales claras en {symbol}. Mantener posición.")

# ======= LOOP PRINCIPAL MULTIPLES PARES =======

def main():
    print("🔁 Iniciando bot multi-activos con estrategias combinadas...\n")

    pares = obtener_pares_disponibles()
    activos = obtener_activos_con_saldo()
    pares_validos = pares_usdt_activos(activos, pares)

    print(f"Monedas con saldo y par USDT disponibles: {pares_validos}")

    while True:
        try:
            for symbol in pares_validos:
                print(f"\nProcesando {symbol}...")

                precios = obtener_precios_historicos(symbol, dias=30)
                if len(precios) < 10:
                    print(f"⚠️ Datos insuficientes para {symbol}. Saltando.")
                    continue
                # Filtrar precios inválidos
                precios = [p for p in precios if p > 0 and np.isfinite(p)]
                if len(set(precios)) < 2:
                    print(f"⚠️ Precios inválidos o constantes para {symbol}. Saltando.")
                    continue
                
                log_precios = np.log(precios)
                shape, loc, scale = lognorm.fit(log_precios, floc=0)

                precio_actual = float(client.get_symbol_ticker(symbol=symbol)["price"])
                decidir_y_actuar(precio_actual, precios, shape, loc, scale, symbol)

            print(f"\n⌛ Esperando {INTERVALO_ESPERA} segundos...\n")
            time.sleep(INTERVALO_ESPERA)

        except Exception as e:
            print("⚠️ Error general:", e)
            time.sleep(10)

if __name__ == "__main__":
    main()


In [ ]:
from strategy import PrecioActual

symbol = "BTCUSDT"
a = PrecioActual(symbol)
print(a.precio())

In [12]:
l = [1,2]
l.append(3)
l.popleft()
print(l)

AttributeError: 'list' object has no attribute 'popleft'